# 02 - Training tomato, pumpkin, lemon

Notebook ini memakai trainer resmi paket. Untuk RTX 2050 4 GB, mulai dari `BATCH = 2`. Jika VRAM penuh, ubah menjadi `1`. Kernel VS Code harus memakai `.venv` yang sudah berisi dependency.

In [ ]:
import subprocess
import sys
from pathlib import Path

PACKAGE = Path.cwd()
PACKAGE = next((p for p in (PACKAGE, *PACKAGE.parents) if (p / 'dataset' / 'data.yaml').is_file()), None) or next((p.parent.parent for p in Path.cwd().glob('**/dataset/data.yaml')), None)
if PACKAGE is None:
    raise FileNotFoundError('Folder paket tomato-pumpkin-lemon-training belum dibuka.')

RUNNER = PACKAGE / 'training' / 'train_auto.py'
EPOCHS = None  # None = 100 untuk CUDA, 10 untuk CPU/AMD
BATCH = None   # None = 2 untuk CUDA, 1 untuk CPU/AMD
RESUME = None  # isi path ke weights/last.pt jika melanjutkan
print('Package:', PACKAGE)
print('Python kernel:', sys.executable)
print('Runner:', RUNNER)
print('Konfigurasi: adaptive, EPOCHS=', EPOCHS, 'BATCH=', BATCH, 'RESUME=', RESUME)


In [ ]:
import importlib.util
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('PyTorch belum terpasang pada kernel ini. Install requirements-training.txt terlebih dahulu.')
print('Hardware akan dideteksi otomatis oleh train_auto.py')


In [ ]:
command = [sys.executable, str(RUNNER)]
if EPOCHS is not None:
    command += ['--epochs', str(EPOCHS)]
if BATCH is not None:
    command += ['--batch', str(BATCH)]
if RESUME is not None:
    command += ['--resume', str(RESUME)]
print('Menjalankan:', ' '.join(command))
result = subprocess.run(command, cwd=PACKAGE)
if result.returncode != 0:
    raise RuntimeError(f'Training gagal dengan exit code {result.returncode}. Jika CUDA out of memory, jalankan ulang dengan --batch 1.')
print('Training selesai atau checkpoint terakhir tersimpan di runs/adaptive/.')
